# Phase 1: Data Factory

In this notebook, we transform the raw **MovieLens 100k** dataset into a structured format for **Link Prediction**. 

### Goals:
1. **Implicit Feedback Transformation**: Convert ratings into binary labels (Rating $\ge$ 4 = Like).
2. **Feature Engineering**: 
    - Multi-hot encoding for movie genres.
    - Categorical encoding for user demographics (Age, Gender, Occupation).
3. **Train/Test Split**: Create an 80/20 split on the graph's edges.
4. **Negative Sampling**: Generate "non-interactions" to train the models on what a "missing link" looks like.
5. **Data Persistence**: Save processed files to `processed_data/` for downstream notebooks.

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

# Ensure the output directory exists
OUTPUT_DIR = 'processed_data/'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Libraries imported and directory ready.")

Libraries imported and directory ready.


## 1. Loading the Data

We load the three core files:
- `u.data`: User-Item interactions.
- `u.item`: Movie details and genres.
- `u.user`: User demographics.

In [2]:
# 1. Load Interactions
df_ratings = pd.read_csv('data/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

# 2. Load Item Info (Genres are the last 19 columns)
genres = [
    "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]
item_cols = ['item_id', 'title', 'release_date', 'video_release_date', 'imdb_url'] + genres
df_items = pd.read_csv('data/u.item', sep='|', names=item_cols, encoding='latin-1')

# 3. Load User Info
df_users = pd.read_csv('data/u.user', sep='|', names=['user_id', 'age', 'gender', 'occupation', 'zip_code'])

print(f"Loaded {len(df_ratings)} ratings, {len(df_items)} items, and {len(df_users)} users.")

Loaded 100000 ratings, 1682 items, and 943 users.


## 2. Filtering & ID Mapping

We need to filter for high ratings and map IDs to start from 0 for matrix compatibility.

In [3]:
# Keep only ratings >= 4 (Implicit Feedback)
df_positive = df_ratings[df_ratings['rating'] >= 4].copy()

# Map User and Item IDs to 0-indexed range
user_map = {old: new for new, old in enumerate(df_users['user_id'].unique())}
item_map = {old: new for new, old in enumerate(df_items['item_id'].unique())}

df_positive['user_idx'] = df_positive['user_id'].map(user_map)
df_positive['item_idx'] = df_positive['item_id'].map(item_map)

num_users = len(user_map)
num_items = len(item_map)

print(f"Filtered for {len(df_positive)} positive interactions.")
print(f"Mapped {num_users} users and {num_items} items to 0-indexed values.")

Filtered for 55375 positive interactions.
Mapped 943 users and 1682 items to 0-indexed values.


## 3. Feature Engineering

We convert genres into a binary matrix and encode user demographics into numeric features.

In [4]:
# Item Features: Extract the genre matrix
item_features = df_items[genres].values.astype(np.float32)

# User Features: Encode Categorical columns
user_data = df_users.copy()
le_gender = LabelEncoder()
user_data['gender'] = le_gender.fit_transform(user_data['gender'])

le_occupation = LabelEncoder()
user_data['occupation'] = le_occupation.fit_transform(user_data['occupation'])

# Normalize Age
user_data['age'] = user_data['age'] / user_data['age'].max()

# Combine into a single matrix
user_features = user_data[['age', 'gender', 'occupation']].values.astype(np.float32)

print(f"Item feature matrix shape: {item_features.shape}")
print(f"User feature matrix shape: {user_features.shape}")

Item feature matrix shape: (1682, 19)
User feature matrix shape: (943, 3)


## 4. Train / Test Split

We split the positive edges into 80% Training and 20% Testing. This split ensures that our models learn from a majority of the graph and are evaluated on unseen edges.

In [5]:
train_df, test_df = train_test_split(df_positive, test_size=0.20, random_state=42, stratify=df_positive['user_idx'])

print(f"Training set size: {len(train_df)} edges.")
print(f"Test set size: {len(test_df)} edges.")

Training set size: 44300 edges.
Test set size: 11075 edges.


## 5. Negative Sampling

For evaluating metrics like AUC, we need "Fake" links (pairs that never occurred). We sample one negative edge for every user interaction.

In [6]:
# Identify all positive interactions to avoid sampling them as negatives
all_positives = set(zip(df_positive['user_idx'], df_positive['item_idx']))

def get_negative_samples(pos_df, num_negatives=1):
    neg_samples = []
    for u in pos_df['user_idx'].unique():
        u_positives = pos_df[pos_df['user_idx'] == u]['item_idx'].values
        count = 0
        while count < num_negatives * len(u_positives):
            i = np.random.randint(0, num_items)
            if (u, i) not in all_positives:
                neg_samples.append([u, i, 0])
                count += 1
    return pd.DataFrame(neg_samples, columns=['user_idx', 'item_idx', 'label'])

print("Generating negative samples for evaluation...")
test_neg_df = get_negative_samples(test_df)
print(f"Generated {len(test_neg_df)} negative samples.")

Generating negative samples for evaluation...
Generated 11075 negative samples.


## 6. Save Processed Data

Finally, we save all matrices and CSVs for use in the next notebooks.

In [7]:
# Save dataframes
train_df[['user_idx', 'item_idx']].to_csv(os.path.join(OUTPUT_DIR, 'train_edges.csv'), index=False)
test_df[['user_idx', 'item_idx']].to_csv(os.path.join(OUTPUT_DIR, 'test_edges.csv'), index=False)
test_neg_df.to_csv(os.path.join(OUTPUT_DIR, 'test_neg_edges.csv'), index=False)

# Save feature matrices
np.save(os.path.join(OUTPUT_DIR, 'user_features.npy'), user_features)
np.save(os.path.join(OUTPUT_DIR, 'item_features.npy'), item_features)

# Save metadata
metadata = {
    'num_users': num_users,
    'num_items': num_items,
    'user_map': user_map,
    'item_map': item_map
}
with open(os.path.join(OUTPUT_DIR, 'metadata.pkl'), 'wb') as f:
    pickle.dump(metadata, f)

print("All data successfully saved to 'processed_data/'!")

All data successfully saved to 'processed_data/'!
